# Object Detection with YOLOS Small (Hugging Face)

This notebook uses the `hustvl/yolos-small` model to detect objects in the images situated within the `images` folder.

In [ ]:
import os
import torch
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
from transformers import YolosImageProcessor, AutoModelForObjectDetection

%matplotlib inline

: 

## 1. Load Model and Processor

In [ ]:
model_checkpoint = "hustvl/yolos-small"
processor = YolosImageProcessor.from_pretrained(model_checkpoint)
model = AutoModelForObjectDetection.from_pretrained(model_checkpoint)
model.eval()

print(f"Model '{model_checkpoint}' loaded successfully.")

## 2. Detection and Professional Visualization Function

In [ ]:
def detect_and_visualize(image_path, threshold=0.75):
    """Runs object detection and plots the result with bounding boxes."""
    image = Image.open(image_path).convert("RGB")
    inputs = processor(images=image, return_tensors="pt")
    
    with torch.no_grad():
        outputs = model(**inputs)
    
    # Post-process detections: filter by threshold
    target_sizes = torch.tensor([image.size[::-1]])
    results = processor.post_process_object_detection(outputs, threshold=threshold, target_sizes=target_sizes)[0]
    
    # Creating the visualization plot
    fig, ax = plt.subplots(figsize=(12, 8))
    ax.imshow(image)
    
    # Filter scores and boxes
    scores = results["scores"]
    labels = results["labels"]
    boxes = results["boxes"]

    for score, label, box in zip(scores, labels, boxes):
        box = [round(i, 2) for i in box.tolist()]
        label_name = model.config.id2label[label.item()]
        
        # Drawing Bounding Box
        x, y, x2, y2 = box
        width, height = x2 - x, y2 - y
        
        rect = patches.Rectangle((x, y), width, height, linewidth=2.5, edgecolor='#12d46e', facecolor='none')
        ax.add_patch(rect)
        
        # Fancy label box
        ax.text(x, y - 10, f"{label_name} ({score:.2%})", 
                fontsize=11, fontweight='bold', color='white', 
                bbox=dict(facecolor='#12d46e', alpha=0.8, edgecolor='none', pad=2, boxstyle='round,pad=0.3'))

    plt.title(f"Detections for: {os.path.basename(image_path)}", fontsize=15, fontweight='bold', pad=20)
    plt.axis("off")
    plt.show()

## 3. Run Inference on Images

In [ ]:
images_dir = 'images'
image_files = [f for f in os.listdir(images_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

if not image_files:
    print(f"No images found in {images_dir}.")
else:
    for filename in image_files:
        image_path = os.path.join(images_dir, filename)
        detect_and_visualize(image_path)